**Documentação do Notebook: Agrupamento2_HDBSCAN**

Este notebook tem como objetivo realizar o pré-processamento e a engenharia de features no restante dos conjuntos de dados que não foram tratados no notebook anterior: Clientes Desde, Contratações 12 Meses, MRR, Notas de Jornada e Tickets (clientes_desde.csv, contratacoes_12meses.csv e mrr.csv, nps_relacional.csv, nps_transacional_aquisicao.csv, nps_transacional_implantacao.csv, nps_transacional_onboarding.csv, nps_transacional_produto.csv, nps_transacional_suporte.csv e tickets.csv).

O processo é o mesmo, e consiste em carregar os dados, limpá-los, transformá-los e agregar as informaçõesm, novamente, em um nível de cliente, preparando-os para serem utilizados em modelos de clusterização. Ao final de cada etapa de processamento, um arquivo Parquet consolidado é salvo.

**Módulo 1: Configuração Inicial e Funções Auxiliares**

Esta seção é responsável por preparar o ambiente de análise de dados.

* Bibliotecas:

  1. pandas: Biblioteca fundamental para manipulação e análise de dados tabulares (DataFrames).

* Configurações de Exibição do Pandas:

  1. display.max_columns e display.max_rows: Configurados como None para garantir que, ao exibir um DataFrame, todas as suas colunas e linhas sejam mostradas, facilitando a inspeção visual completa dos dados.

  2. display.float_format: Formata todos os números de ponto flutuante para exibição com duas casas decimais, melhorando a legibilidade de valores monetários e métricas.

* Função de Leitura de CSV (try_read_csv):

  1. Define uma função robusta para carregar arquivos CSV. Ela tenta ler um arquivo sequencialmente com diferentes codificações (utf-8, latin1) e separadores (;), tratando exceções comuns de formatação. Isso torna o script mais resiliente a variações nos arquivos de origem. Se nenhuma combinação funcionar, a função levanta um erro claro.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
def try_read_csv(path):
    for encoding in ['utf-8', 'latin1']:
        for sep in [';']:
            try:
                return pd.read_csv(path, encoding=encoding, sep=sep)
            except Exception:
                continue
    raise ValueError(f'Erro ao ler o arquivo: {path}')

**Módulo 2: Carregamento dos Dados**

Esta seção é responsável por carregar um conjunto de arquivos de dados menores em volume, mas ricos em informações sobre a jornada e o relacionamento do cliente. A função try_read_csv definida neste script novamente, é reutilizada para garantir um carregamento robusto.

* Fontes de Dados Carregadas:

  1. clientes_desde.csv: Data de início do relacionamento de cada cliente.

  2. contratacoes_ultimos_12_meses.csv: Volume de contratações recentes.

  3. mrr.csv: Dados de Receita Mensal Recorrente.

  4. nps_*.csv: Arquivos distintos com dados de pesquisas de satisfação  (NPS).

  5. tickets.csv: Histórico de tickets geral.

In [ ]:
clientes_desde = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/clientes_desde.csv')
contratacoes_12meses = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/contratacoes_ultimos_12_meses.csv')
mrr = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/mrr.csv')
nps_relacional = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/nps_relacional.csv')
nps_transacional_aquisicao = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/nps_transacional_aquisicao.csv')
nps_transacional_implantacao = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/nps_transacional_implantacao.csv')
nps_transacional_onboarding = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/nps_transacional_onboarding.csv')
nps_transacional_produto = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/nps_transacional_produto.csv')
nps_transacional_suporte = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/nps_transacional_suporte.csv')
tickets = try_read_csv(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Dados/tickets.csv')

**QUARTA PARTE: Processamento do Tempo de Relacionamento (Clientes Desde)**

O objetivo é calcular há quanto tempo cada cliente está na base.

* Tratamento e Engenharia de Features:

  1. A coluna CLIENTE é renomeada para CD_CLIENTE para padronização.

  2. A coluna CLIENTE_DESDE é convertida para o formato datetime.

  3. É definida uma data_referencia ('2025-04-23') fixa, data esta que representa a última atualização dos dados que tivemos na plataforma da FIAP. Utilizar esta data de referência é fundamental para garantir que os cálculos de "idade" do cliente sejam consistentes e reprodutíveis, representando um "retrato" real da base.

  4. Duas novas features são criadas:

    * IDADE_DIAS: O número total de dias desde que o cliente entrou na base até a data de referência.

    * IDADE_ANOS: A mesma medida, convertida para anos.

* Exportação: O DataFrame resultante é salvo como clientes_desde_final.parquet.

In [ ]:
clientes_desde = clientes_desde.rename(columns={
    'CLIENTE': 'CD_CLIENTE'
})

clientes_desde['CLIENTE_DESDE'] = pd.to_datetime(clientes_desde['CLIENTE_DESDE'])
data_referencia = pd.to_datetime('2025-04-23')

clientes_desde['IDADE_DIAS'] = (data_referencia - clientes_desde['CLIENTE_DESDE']).dt.days
clientes_desde['IDADE_ANOS'] = clientes_desde['IDADE_DIAS'] / 365.25

display(clientes_desde.head())

clientes_desde.to_parquet(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/clientes_desde_final.parquet', index=False)

,CD_CLIENTE,CLIENTE_DESDE,IDADE_DIAS,IDADE_ANOS
0,TFDICB,2023-01-27,817,2.24
1,TFCU91,2021-01-26,1548,4.24
2,TFDDYV,2022-07-18,1010,2.77
3,TEZBRW,2022-10-28,908,2.49
4,TEZCXN,2013-05-24,4352,11.92


**QUINTA PARTE: Processamento de Dados Financeiros (Contratações e MRR)**

Esta seção realiza tratamentos simples em dados financeiros para garantir que estejam no formato correto para análise.

* Contratações nos Últimos 12 Meses

  1. Objetivo: Tratar os valores de contratações recentes.

  2. Tratamento: A coluna VLR_CONTRATACOES_12M, que contém valores numéricos com vírgula decimal, é convertida para o tipo float.

  3. Exportação: O DataFrame limpo é salvo como contratacoes_12meses_final.parquet.

* Receita Mensal Recorrente (MRR)

  1. Objetivo: Padronizar o arquivo de MRR.

  2. Tratamento: Apenas a coluna CLIENTE é renomeada para CD_CLIENTE para consistência. Nenhum outro tratamento é necessário.

* Exportação: O DataFrame é salvo como mrr_final.parquet.

**CONTRATACOES 12 MESES:** Informações sobre contratações recentes.

In [ ]:
contratacoes_12meses['VLR_CONTRATACOES_12M'] = contratacoes_12meses['VLR_CONTRATACOES_12M'].str.replace(',', '.').astype(float)

In [ ]:
contratacoes_12meses.head()

,CD_CLIENTE,QTD_CONTRATACOES_12M,VLR_CONTRATACOES_12M
0,T07544,1,2104.64
1,T01872,2,0.01
2,T05174,3,1777.91
3,T01670,2,2934.86
4,T02817,2,4207.22


In [ ]:
contratacoes_12meses.to_parquet(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/contratacoes_12meses_final.parquet', index=False)

**MRR:** Dados de receita mensal recorrente.

In [ ]:
mrr = mrr.rename(columns={'CLIENTE': 'CD_CLIENTE'})
mrr.head()

,CD_CLIENTE,MRR_12M
0,T03360,485.25
1,T01872,287.07
2,T01899,628.72
3,T01670,207.50
4,T02817,890.03


In [ ]:
mrr.to_parquet(r'/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/mrr_final.parquet', index=False)

**SEXTA PARTE: Processamento dos Dados de NPS (Net Promoter Score)**

Esta é uma seção complexa que visa consolidar, padronizar e agregar os múltiplos arquivos de NPS, que medem a satisfação do cliente em diferentes pontos de contato.

* Padronização de Colunas
  1. Desafio: Os arquivos de NPS originais possuem nomes de colunas inconsistentes, longos e, por vezes, em português com acentos e caracteres especiais.

  2. Solução: Uma etapa crucial de pré-processamento é realizada onde as colunas de todos os DataFrames de NPS são renomeadas para um padrão único, curto e sem caracteres especiais (ex: Cód. Cliente para CD_CLIENTE, Data da Resposta para DT_RESPOSTA). Isso é vital para facilitar as agregações e futuras junções de dados.

* Agregação dos NPS (Relacional e Transacionais Padrão)
  1. Objetivo: Calcular a nota média de satisfação por cliente para cada tipo de pesquisa.

  2. Método: Para os NPS relacional, transacional_aquisicao, transacional_implantacao e transacional_onboarding, é aplicada uma agregação simples.

    * Os dados são agrupados por CD_CLIENTE.

    * A média de cada nota de avaliação é calculada. O resultado é um DataFrame para cada tipo de NPS, onde cada linha representa um cliente e as colunas representam sua nota média em cada quesito avaliado.

* Exportação: Cada um desses DataFrames agregados é salvo em seu próprio arquivo Parquet.

In [ ]:
nps_relacional = nps_relacional.rename(columns={
    'respondedAt': 'DT_RESPOSTA',
    'metadata_codcliente': 'CD_CLIENTE',
    'resposta_NPS': 'RESPOSTA_NPS',
    'resposta_unidade': 'RESPOSTA_UNIDADE',
    'Nota_SupTec_Agilidade': 'NOTA_SUPTEC_AGI',
    'Nota_SupTec_Atendimento': 'NOTA_SUPTEC_ATEN',
    'Nota_Comercial': 'NOTA_COMERCIAL',
    'Nota_Custos': 'NOTA_CUSTOS',
    'Nota_AdmFin_Atendimento': 'NOTA_ADMFIN_ATEN',
    'Nota_Software': 'NOTA_SOFTWARE',
    'Nota_Software_Atualizacao': 'NOTA_SOFTWARE_ATT'
})

nps_transacional_aquisicao = nps_transacional_aquisicao.rename(columns={
    'Cód. Cliente': 'CD_CLIENTE',
    'Data da Resposta': 'DT_RESPOSTA',
    'Nota NPS': 'NOTA_NPS',
    'Nota Agilidade': 'NOTA_AGILIDADE',
    'Nota Conhecimento': 'NOTA_CONHECIMENTO',
    'Nota Custo': 'NOTA_CUSTO',
    'Nota Facilidade': 'NOTA_FACILIDADE',
    'Nota Flexibilidade': 'NOTA_FLEXIBILIDADE'
})

nps_transacional_implantacao = nps_transacional_implantacao.rename(columns={
    'Cód. Cliente': 'CD_CLIENTE',
    'Data da Resposta': 'DT_RESPOSTA',
    'Nota NPS': 'NOTA_NPS',
    'Nota Metodologia': 'NOTA_METODOLOGIA',
    'Nota Gestao': 'NOTA_GESTAO',
    'Nota Conhecimento': 'NOTA_CONHECIMENTO',
    'Nota Qualidade': 'NOTA_QUALIDADE',
    'Nota Comunicacao': 'NOTA_COMUNICACAO',
    'Nota Prazos': 'NOTA_PRAZOS'
})

nps_transacional_onboarding = nps_transacional_onboarding.rename(columns={
    'Cod Cliente': 'CD_CLIENTE',
    'Data de Resposta': 'DT_RESPOSTA',
    'Em uma escala de 0 a 10, quanto você recomenda o Onboarding da TOTVS para um amigo ou colega?.': 'NOTA_RECOMENDACAO',
    'Em uma escala de 0 a 10, o quanto você acredita que o atendimento CS Onboarding ajudou no início da sua trajetória com a TOTVS?': 'NOTA_AJUDA',
    '- Duração do tempo na realização da reunião de Onboarding;': 'NOTA_TEMPO',
    '- Clareza no acesso aos canais de comunicação da TOTVS;': 'NOTA_CLAREZA_CANAL',
    '- Clareza nas informações em geral transmitidas pelo CS que lhe atendeu no Onboarding;': 'NOTA_CLAREZA_GERAL',
    '- Expectativas atendidas no Onboarding da TOTVS.': 'NOTA_EXPECTATIVA'
})

nps_transacional_produto = nps_transacional_produto.rename(columns={
    'Cód. T': 'CD_CLIENTE',
    'Data da Resposta': 'DT_RESPOSTA',
    'Linha de Produto': 'LINHA_PRODUTO',
    'Nome do Produto': 'NM_PRODUTO',
    'Nota': 'NOTA_PRODUTO'
})

nps_transacional_suporte = nps_transacional_suporte.rename(columns={
    'cliente': 'CD_CLIENTE',
    'ticket': 'TICKET',
    'resposta_NPS': 'NOTA_NPS',
    'grupo_NPS': 'GRUPO_NPS',
    'Nota_ConhecimentoAgente': 'NOTA_CONHECIMENTO_AG',
    'Nota_Solucao': 'NOTA_SOLUCAO',
    'Nota_TempoRetorno': 'NOTA_TEMPO_RETORNO',
    'Nota_Facilidade': 'NOTA_FACILIDADE',
    'Nota_Satisfacao': 'NOTA_SATISFACAO'
})

In [ ]:
nps_relacional_agregado = nps_relacional.groupby('CD_CLIENTE').agg(
    RELACIONAL_MEDIA_RESPOSTA_NPS=('RESPOSTA_NPS', 'mean'),
    RELACIONAL_MEDIA_RESPOSTA_UNIDADE=('RESPOSTA_UNIDADE', 'mean'),
    RELACIONAL_MEDIA_NOTA_SUPTEC_AGI=('NOTA_SUPTEC_AGI', 'mean'),
    RELACIONAL_MEDIA_NOTA_SUPTEC_ATEN=('NOTA_SUPTEC_ATEN', 'mean'),
    RELACIONAL_MEDIA_NOTA_COMERCIAL=('NOTA_COMERCIAL', 'mean'),
    RELACIONAL_MEDIA_NOTA_CUSTOS=('NOTA_CUSTOS', 'mean'),
    RELACIONAL_MEDIA_NOTA_ADMFIN_ATEN=('NOTA_ADMFIN_ATEN', 'mean'),
    RELACIONAL_MEDIA_NOTA_SOFTWARE=('NOTA_SOFTWARE', 'mean'),
    RELACIONAL_MEDIA_NOTA_SOFTWARE_ATT=('NOTA_SOFTWARE_ATT', 'mean')
).reset_index()


nps_transacional_aquisicao_agregado = nps_transacional_aquisicao.groupby('CD_CLIENTE').agg(
    AQUISICAO_MEDIA_NOTA_NPS=('NOTA_NPS', 'mean'),
    AQUISICAO_MEDIA_NOTA_AGILIDADE=('NOTA_AGILIDADE', 'mean'),
    AQUISICAO_MEDIA_NOTA_CONHECIMENTO=('NOTA_CONHECIMENTO', 'mean'),
    AQUISICAO_MEDIA_NOTA_CUSTO=('NOTA_CUSTO', 'mean'),
    AQUISICAO_MEDIA_NOTA_FACILIDADE=('NOTA_FACILIDADE', 'mean'),
    AQUISICAO_MEDIA_NOTA_FLEXIBILIDADE=('NOTA_FLEXIBILIDADE', 'mean')
).reset_index()


nps_transacional_implantacao_agregado = nps_transacional_implantacao.groupby('CD_CLIENTE').agg(
    IMPLANTACAO_MEDIA_NOTA_NPS=('NOTA_NPS', 'mean'),
    IMPLANTACAO_MEDIA_NOTA_METODOLOGIA=('NOTA_METODOLOGIA', 'mean'),
    IMPLANTACAO_MEDIA_NOTA_GESTAO=('NOTA_GESTAO', 'mean'),
    IMPLANTACAO_MEDIA_NOTA_CONHECIMENTO=('NOTA_CONHECIMENTO', 'mean'),
    IMPLANTACAO_MEDIA_NOTA_QUALIDADE=('NOTA_QUALIDADE', 'mean'),
    IMPLANTACAO_MEDIA_NOTA_COMUNICACAO=('NOTA_COMUNICACAO', 'mean'),
    IMPLANTACAO_MEDIA_NOTA_PRAZOS=('NOTA_PRAZOS', 'mean')
).reset_index()


nps_transacional_onboarding_agregado = nps_transacional_onboarding.groupby('CD_CLIENTE').agg(
    ONBOARDING_MEDIA_NOTA_RECOMENDACAO=('NOTA_RECOMENDACAO', 'mean'),
    ONBOARDING_MEDIA_NOTA_AJUDA=('NOTA_AJUDA', 'mean'),
    ONBOARDING_MEDIA_NOTA_TEMPO=('NOTA_TEMPO', 'mean'),
    ONBOARDING_MEDIA_NOTA_CLAREZA_CANAL=('NOTA_CLAREZA_CANAL', 'mean'),
    ONBOARDING_MEDIA_NOTA_CLAREZA_GERAL=('NOTA_CLAREZA_GERAL', 'mean'),
    ONBOARDING_MEDIA_NOTA_EXPECTATIVA=('NOTA_EXPECTATIVA', 'mean')
).reset_index()

In [ ]:
nps_relacional_agregado.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/nps_relacional_agregado_final.parquet', index=False)
nps_transacional_aquisicao_agregado.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/nps_transacional_aquisicao_agregado_final.parquet', index=False)
nps_transacional_implantacao_agregado.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/nps_transacional_implantacao_agregado_final.parquet', index=False)
nps_transacional_onboarding_agregado.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/nps_transacional_onboarding_agregado_final.parquet', index=False)

**Tratamento Específico: NPS de Suporte**

Particularidade: Este DataFrame contém, além das notas, a classificação do cliente em grupos NPS (Promotor, Detrator, Neutro).

* Agregação Mista:

  1. Notas Médias: As notas de avaliação (NOTA_NPS, NOTA_SOLUCAO, etc.) são agregadas pela média, e a quantidade de tickets únicos (QTD_TICKET_PARA_SUPORTE) é contada.

  2. Contagem de Grupos NPS (Pivotagem): A técnica groupby().value_counts().unstack() é utilizada para contar, para cada cliente, quantas vezes ele foi classificado em cada grupo. Isso cria novas colunas como GRUPO_NPS_Promotor, GRUPO_NPS_Detrator, etc.

Consolidação e Exportação: Os dois resultados são unidos, criando um perfil rico do cliente no suporte, que é salvo como nps_transacional_suporte_final.parquet.

In [ ]:
nps_transacional_suporte.head()

,TICKET,NOTA_NPS,GRUPO_NPS,NOTA_CONHECIMENTO_AG,NOTA_SOLUCAO,NOTA_TEMPO_RETORNO,NOTA_FACILIDADE,NOTA_SATISFACAO,CD_CLIENTE
0,19478703,10,promotor,10.00,10.00,10.00,10.00,10.00,TFCLHU
1,19994447,9,promotor,9.00,9.00,9.00,9.00,9.00,TFCDXR
2,19467742,10,promotor,NaN,NaN,NaN,NaN,NaN,T16451
3,18992197,10,promotor,NaN,NaN,NaN,NaN,NaN,TEZJP5
4,19590718,10,promotor,10.00,10.00,10.00,10.00,10.00,TFCNRP


In [ ]:
valor_grupo = (
    nps_transacional_suporte
    .groupby('CD_CLIENTE')['GRUPO_NPS']
    .value_counts()
    .unstack(fill_value=0)
)

valor_grupo.columns = ['GRUPO_NPS_' + str(col) for col in valor_grupo.columns]

nps_transacional_suporte_agregado = nps_transacional_suporte.groupby('CD_CLIENTE').agg(
    SUPORTE_MEDIA_NOTA_NPS=('NOTA_NPS', 'mean'),
    SUPORTE_MEDIA_NOTA_CONHECIMENTO_AG=('NOTA_CONHECIMENTO_AG', 'mean'),
    SUPORTE_MEDIA_NOTA_SOLUCAO=('NOTA_SOLUCAO', 'mean'),
    SUPORTE_MEDIA_NOTA_TEMPO_RETORNO=('NOTA_TEMPO_RETORNO', 'mean'),
    SUPORTE_MEDIA_NOTA_FACILIDADE=('NOTA_FACILIDADE', 'mean'),
    SUPORTE_MEDIA_NOTA_SATISFACAO=('NOTA_SATISFACAO', 'mean'),

    QTD_TICKET_PARA_SUPORTE=('TICKET', 'nunique'),
).reset_index()

nps_transacional_suporte_final = pd.merge(
    nps_transacional_suporte_agregado,
    valor_grupo,
    on='CD_CLIENTE',
    how='left'
)

In [ ]:
nps_transacional_suporte_final.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/nps_transacional_suporte_final.parquet', index=False)

**Tratamento Específico: NPS de Produto**

Particularidade: Este dataset avalia a satisfação por linha de produto, exigindo uma agregação mais granular.

* Agregação em Duas Frentes (Pivotagem):

  1. Contagem por Linha de Produto: Primeiro, o código cria uma matriz (matriz_linha_produto) onde as linhas são clientes, as colunas são as diferentes LINHA_PRODUTO, e os valores representam a quantidade de avaliações que o cliente fez para cada linha.

  2. Nota Média por Linha de Produto: Em seguida, outra matriz (matriz_nota_linha_produto) é criada usando .pivot(). A estrutura é a mesma, mas os valores representam a nota média que o cliente deu para cada linha de produto.

Consolidação e Exportação: As duas matrizes são unidas para criar um DataFrame final que mostra, para cada cliente, tanto a quantidade de produtos avaliados quanto a satisfação média por linha. O resultado é salvo como nps_transacional_produto_agregado_final.parquet.

In [ ]:
matriz_linha_produto = (
    nps_transacional_produto
    .groupby('CD_CLIENTE')['LINHA_PRODUTO']
    .value_counts()
    .unstack(fill_value=0)
)

matriz_linha_produto.columns = ['LINHA_PRODUTO_' + str(col) for col in matriz_linha_produto.columns]

In [ ]:
nota_por_produto = (
    nps_transacional_produto
    .groupby(['CD_CLIENTE', 'LINHA_PRODUTO'])['NOTA_PRODUTO']
    .mean()
    .reset_index()
)

nota_por_produto.head()

,CD_CLIENTE,LINHA_PRODUTO,NOTA_PRODUTO
0,000001,Protheus,8.99
1,000001,RM,7.40
2,000001,TOTVS Agro Multicultivo,10.00
3,000002,Protheus,7.47
4,000002,RM,9.00


In [ ]:
matriz_nota_linha_produto = nota_por_produto.pivot(index='CD_CLIENTE', columns='LINHA_PRODUTO', values='NOTA_PRODUTO').fillna(0)
matriz_nota_linha_produto.columns = ['LINHA_PRODUTO_' + str(col) + '_MEDIA_NOTA' for col in matriz_nota_linha_produto.columns]
matriz_nota_linha_produto.reset_index(inplace=True)
matriz_nota_linha_produto.head()

,CD_CLIENTE,LINHA_PRODUTO_Consinco_MEDIA_NOTA,LINHA_PRODUTO_Datasul_MEDIA_NOTA,LINHA_PRODUTO_Protheus_MEDIA_NOTA,LINHA_PRODUTO_RM_MEDIA_NOTA,LINHA_PRODUTO_SARA_MEDIA_NOTA,LINHA_PRODUTO_TOTVS Agro Bioenergia_MEDIA_NOTA,LINHA_PRODUTO_TOTVS Agro Multicultivo_MEDIA_NOTA,LINHA_PRODUTO_TOTVS Moda_MEDIA_NOTA
0,000001,0.00,0.00,8.99,7.40,0.00,0.00,10.00,0.00
1,000002,0.00,0.00,7.47,9.00,0.00,0.00,0.00,0.00
2,000003,0.00,10.00,9.50,9.88,0.00,0.00,0.00,0.00
3,000006,0.00,10.00,9.40,10.00,0.00,0.00,0.00,0.00
4,000008,0.00,0.00,6.62,0.00,0.00,0.00,5.67,0.00


In [ ]:
matriz_linha_produto = (
    nps_transacional_produto
    .groupby('CD_CLIENTE')['LINHA_PRODUTO']
    .value_counts()
    .unstack(fill_value=0)
)

matriz_linha_produto.columns = ['LINHA_PRODUTO_' + str(col) + '_QTD' for col in matriz_linha_produto.columns]
matriz_linha_produto.reset_index(inplace=True)
matriz_linha_produto.head()

,CD_CLIENTE,LINHA_PRODUTO_Consinco_QTD,LINHA_PRODUTO_Datasul_QTD,LINHA_PRODUTO_Protheus_QTD,LINHA_PRODUTO_RM_QTD,LINHA_PRODUTO_SARA_QTD,LINHA_PRODUTO_TOTVS Agro Bioenergia_QTD,LINHA_PRODUTO_TOTVS Agro Multicultivo_QTD,LINHA_PRODUTO_TOTVS Moda_QTD
0,000001,0,0,81,5,0,0,1,0
1,000002,0,0,15,2,0,0,0,0
2,000003,0,4,8,8,0,0,0,0
3,000006,0,1,15,2,0,0,0,0
4,000008,0,0,13,0,0,0,15,0


In [ ]:
nps_transacional_produto_agregado_final = pd.merge(
    matriz_linha_produto,
    matriz_nota_linha_produto,
    on='CD_CLIENTE',
    how='left'
)

In [ ]:
nps_transacional_produto_agregado_final.head()

,CD_CLIENTE,LINHA_PRODUTO_Consinco_QTD,LINHA_PRODUTO_Datasul_QTD,LINHA_PRODUTO_Protheus_QTD,LINHA_PRODUTO_RM_QTD,LINHA_PRODUTO_SARA_QTD,LINHA_PRODUTO_TOTVS Agro Bioenergia_QTD,LINHA_PRODUTO_TOTVS Agro Multicultivo_QTD,LINHA_PRODUTO_TOTVS Moda_QTD,LINHA_PRODUTO_Consinco_MEDIA_NOTA,LINHA_PRODUTO_Datasul_MEDIA_NOTA,LINHA_PRODUTO_Protheus_MEDIA_NOTA,LINHA_PRODUTO_RM_MEDIA_NOTA,LINHA_PRODUTO_SARA_MEDIA_NOTA,LINHA_PRODUTO_TOTVS Agro Bioenergia_MEDIA_NOTA,LINHA_PRODUTO_TOTVS Agro Multicultivo_MEDIA_NOTA,LINHA_PRODUTO_TOTVS Moda_MEDIA_NOTA
0,000001,0,0,81,5,0,0,1,0,0.00,0.00,8.99,7.40,0.00,0.00,10.00,0.00
1,000002,0,0,15,2,0,0,0,0,0.00,0.00,7.47,9.00,0.00,0.00,0.00,0.00
2,000003,0,4,8,8,0,0,0,0,0.00,10.00,9.50,9.88,0.00,0.00,0.00,0.00
3,000006,0,1,15,2,0,0,0,0,0.00,10.00,9.40,10.00,0.00,0.00,0.00,0.00
4,000008,0,0,13,0,0,0,15,0,0.00,0.00,6.62,0.00,0.00,0.00,5.67,0.00


In [ ]:
nps_transacional_produto_agregado_final.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/nps_transacional_produto_agregado_final.parquet', index=False)

**SÉTIMA PARTE: Processamento dos Tickets**

O objetivo aqui é extrair features que descrevam o comportamento do cliente em relação ao acionamento em que tickets são necessários.

* Tratamento e Engenharia de Features:

  1. A coluna de identificação é padronizada para CD_CLIENTE.

  2. As colunas de data são convertidas para o formato datetime.

  3. Uma nova feature, DIAS_DESDE_ATUALIZACAO, é calculada como a diferença em dias entre a criação e a última atualização de um ticket, servindo como um proxy para o tempo de resolução.

* Agregação e Pivotagem:

  1. Estatísticas Gerais: O código primeiro agrega os dados por CD_CLIENTE para calcular: QTD_TICKETS totais, datas do primeiro e último ticket, e estatísticas (média, mín, máx) sobre o tempo de resolução.

  2. Contagem por Categoria (Pivotagem): A função pd.crosstab é usada para contar a ocorrência de cada categoria de TIPO_TICKET, STATUS_TICKET, e PRIORIDADE_TICKET, transformando essas variáveis categóricas em features numéricas (ex: QTD_TIPO_DUVIDA, QTD_STATUS_RESOLVIDO, etc.).

* Consolidação e Exportação: Todas as informações agregadas e pivotadas são unidas em um único DataFrame (df_final), que é então salvo como tickets_agrupado_final.parquet.

In [ ]:
tickets.head()

,CODIGO_ORGANIZACAO,NOME_GRUPO,TIPO_TICKET,STATUS_TICKET,DT_CRIACAO,DT_ATUALIZACAO,BK_TICKET,PRIORIDADE_TICKET
0,TFCPWG,Fábrica Consinco,task,closed,2024-11-12,2025-01-02,21787048,low
1,TB2434,PC - Financeiro,question,closed,2024-05-07,2024-05-16,20066662,normal
2,TFBYNF,TOTVS Chef Corporativo,question,closed,2024-03-19,2024-04-03,19662250,low
3,TFCNAD,PC - Financeiro,question,closed,2024-06-25,2024-07-04,20517470,normal
4,TFDJLF,AGT CLOUD RM STD,question,closed,2024-02-02,2024-02-06,19222316,low


In [ ]:
tickets = tickets.rename(columns={
    'CODIGO_ORGANIZACAO': 'CD_CLIENTE',
})

tickets['DT_ATUALIZACAO'] = pd.to_datetime(tickets['DT_ATUALIZACAO'])
tickets['DT_CRIACAO'] = pd.to_datetime(tickets['DT_CRIACAO'])

tickets['DIAS_DESDE_ATUALIZACAO'] = (tickets['DT_ATUALIZACAO'] - tickets['DT_CRIACAO']).dt.days

agrupado = tickets.groupby('CD_CLIENTE').agg(
    QTD_TICKETS=('BK_TICKET', 'count'),
    PRIMEIRA_CRIACAO=('DT_CRIACAO', 'min'),
    ULTIMA_ATUALIZACAO=('DT_ATUALIZACAO', 'max'),
    DIAS_DESDE_ULTIMA_ATUALIZACAO_MEAN=('DIAS_DESDE_ATUALIZACAO', 'mean'),
    DIAS_DESDE_ULTIMA_ATUALIZACAO_MIN=('DIAS_DESDE_ATUALIZACAO', 'min'),
    DIAS_DESDE_ULTIMA_ATUALIZACAO_MAX=('DIAS_DESDE_ATUALIZACAO', 'max'),
)

tipo_ticket = pd.crosstab(tickets['CD_CLIENTE'], tickets['TIPO_TICKET'])
tipo_ticket.columns = ['QTD_TIPO_' + str(col).upper() for col in tipo_ticket.columns]

status_ticket = pd.crosstab(tickets['CD_CLIENTE'], tickets['STATUS_TICKET'])
status_ticket.columns = ['QTD_STATUS_' + str(col).upper() for col in status_ticket.columns]

prioridade_ticket = pd.crosstab(tickets['CD_CLIENTE'], tickets['PRIORIDADE_TICKET'])
prioridade_ticket.columns = ['QTD_PRIORIDADE_' + str(col).upper() for col in prioridade_ticket.columns]

df_final = agrupado.join([tipo_ticket, status_ticket, prioridade_ticket]).reset_index()

df_final.head()

,CD_CLIENTE,QTD_TICKETS,PRIMEIRA_CRIACAO,ULTIMA_ATUALIZACAO,DIAS_DESDE_ULTIMA_ATUALIZACAO_MEAN,DIAS_DESDE_ULTIMA_ATUALIZACAO_MIN,DIAS_DESDE_ULTIMA_ATUALIZACAO_MAX,QTD_TIPO_INCIDENT,QTD_TIPO_PROBLEM,QTD_TIPO_QUESTION,QTD_TIPO_TASK,QTD_STATUS_CLOSED,QTD_STATUS_DELETED,QTD_STATUS_HOLD,QTD_STATUS_NEW,QTD_STATUS_OPEN,QTD_STATUS_PENDING,QTD_STATUS_SOLVED,QTD_PRIORIDADE_HIGH,QTD_PRIORIDADE_LOW,QTD_PRIORIDADE_NORMAL,QTD_PRIORIDADE_URGENT
0,T00018,2,2025-02-07,2025-03-10,15.50,0,31,0,0,2,0,2,0,0,0,0,0,0,0,2,0,0
1,T00053,326,2024-01-02,2025-03-25,11.93,0,112,3,3,319,1,314,0,3,0,1,0,8,18,198,110,0
2,T00082,261,2024-01-03,2025-03-24,12.40,0,140,0,1,246,14,249,0,6,0,0,1,5,5,149,106,1
3,T00145,54,2024-04-09,2025-03-25,11.37,0,33,1,0,44,9,46,0,0,0,1,3,4,5,31,18,0
4,T00151,68,2024-01-15,2025-03-17,4.65,0,115,0,0,68,0,67,0,0,0,1,0,0,0,30,38,0


In [ ]:
df_final.to_parquet('/content/drive/MyDrive/Rafa_Casa/TOTVS/Modelo_3/Dados_Refinados3/tickets_agrupado_final.parquet', index=False)